# Video Reproduction: F-INR vs Baseline

This notebook runs a representative video encoding experiment on the bundled asset `assets/videos/video.mp4`.

It compares the clean tensor-decomposed video F-INR with the clean baseline spatiotemporal MLP using the same JSON configs exposed by the public script.

In [ ]:
import importlib
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for entry in (ROOT, ROOT / "scripts", ROOT / "src"):
    if str(entry) not in sys.path:
        sys.path.insert(0, str(entry))

import f_inr_video  # noqa: E402

importlib.reload(f_inr_video)
load_video = f_inr_video.load_video
run_video_experiment = f_inr_video.run_video_experiment

finr_config = json.load((ROOT / "configs/video/ground_truth_finr.json").open())
baseline_config = json.load((ROOT / "configs/video/ground_truth_baseline.json").open())
print(f_inr_video.__file__)
video = load_video(finr_config)
video.shape

In [ ]:
frame0 = np.transpose(video[:, 0], (1, 2, 0))
plt.figure(figsize=(5, 5))
plt.imshow(frame0)
plt.title(f"Asset first frame: {Path(finr_config['video_path'])}")
plt.axis("off")
plt.show()

In [ ]:
finr_summary, _ = run_video_experiment(finr_config)
baseline_summary, _ = run_video_experiment(baseline_config)
summaries = [finr_summary, baseline_summary]
print(json.dumps(summaries, indent=2))
print(
    "Speedup (baseline / F-INR):",
    round(baseline_summary["elapsed_seconds"] / finr_summary["elapsed_seconds"], 3),
)

In [ ]:
finr_recon = np.load(Path(finr_summary["output_dir"]) / "reconstruction.npy")
baseline_recon = np.load(Path(baseline_summary["output_dir"]) / "reconstruction.npy")
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(np.transpose(video[:, 0], (1, 2, 0)))
axes[0].set_title("Input frame")
axes[1].imshow(np.transpose(finr_recon[:, 0], (1, 2, 0)))
axes[1].set_title("F-INR frame")
axes[2].imshow(np.transpose(baseline_recon[:, 0], (1, 2, 0)))
axes[2].set_title("Baseline frame")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()